### **Imports and Coneection**

In [2]:
import sys
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Make project root importable
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Now we can import our utilities
from src.utils.logger import get_logger
logger = get_logger("feature_engineering_dev")

DB_PATH = PROJECT_ROOT / "data" / "raw" / "database.sqlite"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PRODUCTION_DIR = PROJECT_ROOT / "data" / "production"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PRODUCTION_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"DB exists: {DB_PATH.exists()}")

2026-04-20 02:03:21 | INFO     | feature_engineering_dev | Project root: /home/muhammed786fiyas/Desktop/Projects/ml_ops/final_project/mlops_end_to_end_project
2026-04-20 02:03:21 | INFO     | feature_engineering_dev | DB exists: True


### **Function 1 — load_raw_data()**

In [3]:
def load_raw_data(db_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load match, team, and team-attribute data from SQLite.
    
    Args:
        db_path: Path to the SQLite database file.
    
    Returns:
        Tuple of (matches_df, teams_df, team_attrs_df).
    """
    logger.info(f"Loading raw data from {db_path}")
    conn = sqlite3.connect(db_path)
    
    # Match table — only the columns we need (10 of 115)
    matches_query = """
        SELECT 
            id, country_id, league_id, season, stage, date,
            home_team_api_id, away_team_api_id,
            home_team_goal, away_team_goal
        FROM Match
        ORDER BY date
    """
    matches = pd.read_sql_query(matches_query, conn)
    matches['date'] = pd.to_datetime(matches['date'])
    logger.info(f"  Loaded {len(matches):,} matches")
    
    # Team table — for name lookups + the FIFA join key
    teams_query = """
        SELECT team_api_id, team_fifa_api_id, team_long_name, team_short_name
        FROM Team
    """
    teams = pd.read_sql_query(teams_query, conn)
    logger.info(f"  Loaded {len(teams):,} teams")
    
    # Team_Attributes — only the 9 numeric FIFA columns + join keys + date
    team_attrs_query = """
        SELECT 
            team_api_id, team_fifa_api_id, date,
            buildUpPlaySpeed, buildUpPlayDribbling, buildUpPlayPassing,
            chanceCreationPassing, chanceCreationCrossing, chanceCreationShooting,
            defencePressure, defenceAggression, defenceTeamWidth
        FROM Team_Attributes
        ORDER BY team_api_id, date
    """
    team_attrs = pd.read_sql_query(team_attrs_query, conn)
    team_attrs['date'] = pd.to_datetime(team_attrs['date'])
    logger.info(f"  Loaded {len(team_attrs):,} team attribute snapshots")
    
    conn.close()
    return matches, teams, team_attrs

In [4]:
matches, teams, team_attrs = load_raw_data(DB_PATH)

print("\n--- Matches sample ---")
print(matches.head(3))
print(f"\nDtypes: {matches.dtypes.tolist()}")

print("\n--- Teams sample ---")
print(teams.head(3))

print("\n--- Team Attributes sample ---")
print(team_attrs.head(3))

2026-04-20 02:06:14 | INFO     | feature_engineering_dev | Loading raw data from /home/muhammed786fiyas/Desktop/Projects/ml_ops/final_project/mlops_end_to_end_project/data/raw/database.sqlite
2026-04-20 02:06:14 | INFO     | feature_engineering_dev |   Loaded 25,979 matches
2026-04-20 02:06:14 | INFO     | feature_engineering_dev |   Loaded 299 teams
2026-04-20 02:06:14 | INFO     | feature_engineering_dev |   Loaded 1,458 team attribute snapshots

--- Matches sample ---
      id  country_id  league_id     season  stage       date  \
0  24559       24558      24558  2008/2009      1 2008-07-18   
1  24560       24558      24558  2008/2009      1 2008-07-19   
2  24561       24558      24558  2008/2009      1 2008-07-20   

   home_team_api_id  away_team_api_id  home_team_goal  away_team_goal  
0             10192              9931               1               2  
1              9930             10179               3               1  
2             10199              9824              

### **Function 2 — derive_target()**

In [ ]:
# Class encoding — keep this as a module-level constant so it's defensible and reusable
OUTCOME_TO_INT = {'H': 0, 'D': 1, 'A': 2}
INT_TO_OUTCOME = {v: k for k, v in OUTCOME_TO_INT.items()}


def derive_target(matches: pd.DataFrame) -> pd.DataFrame:
    """
    Add target columns to the matches DataFrame.
    
    Adds:
        outcome: 'H' (home win), 'D' (draw), or 'A' (away win)
        outcome_encoded: 0 for H, 1 for D, 2 for A (LightGBM-ready)
    
    Args:
        matches: DataFrame with home_team_goal, away_team_goal columns.
    
    Returns:
        Same DataFrame with two new columns added.
    """
    logger.info("Deriving target variable (H/D/A)")
    
    # Vectorized derivation — much faster than .apply() with a lambda
    # np.select reads top-to-bottom: first matching condition wins
    conditions = [
        matches['home_team_goal'] > matches['away_team_goal'],
        matches['home_team_goal'] < matches['away_team_goal'],
    ]
    choices = ['H', 'A']
    matches = matches.copy()  # Don't mutate the input 
    matches['outcome'] = np.select(conditions, choices, default='D')
    matches['outcome_encoded'] = matches['outcome'].map(OUTCOME_TO_INT)
    
    # Sanity check — confirm no nulls slipped in
    null_count = matches['outcome'].isna().sum()
    if null_count > 0:
        logger.error(f"Found {null_count} matches with null outcomes!")
        raise ValueError(f"Target derivation produced {null_count} nulls — investigate")
    
    # Log the distribution so we can verify it matches what we saw in M1.2
    dist = matches['outcome'].value_counts(normalize=True).round(3).to_dict()
    logger.info(f"  Outcome distribution: {dist}")
    
    return matches

In [7]:
matches = derive_target(matches)

print("\n--- Sample with target ---")
print(matches[['date', 'home_team_goal', 'away_team_goal', 'outcome', 'outcome_encoded']].head(10))

print("\n--- Class counts ---")
print(matches['outcome'].value_counts())

print("\n--- Encoding map check ---")
print("OUTCOME_TO_INT:", OUTCOME_TO_INT)
print("INT_TO_OUTCOME:", INT_TO_OUTCOME)

2026-04-20 02:13:48 | INFO     | feature_engineering_dev | Deriving target variable (H/D/A)
2026-04-20 02:13:48 | INFO     | feature_engineering_dev |   Outcome distribution: {'H': 0.459, 'A': 0.287, 'D': 0.254}

--- Sample with target ---
        date  home_team_goal  away_team_goal outcome  outcome_encoded
0 2008-07-18               1               2       A                2
1 2008-07-19               3               1       H                0
2 2008-07-20               1               2       A                2
3 2008-07-20               1               2       A                2
4 2008-07-23               1               0       H                0
5 2008-07-23               1               2       A                2
6 2008-07-23               1               0       H                0
7 2008-07-24               2               1       H                0
8 2008-07-24               0               2       A                2
9 2008-07-26               2               0       H        

### **Function 3 — compute_rolling_form()**

In [8]:
def _build_team_centric_view(matches: pd.DataFrame) -> pd.DataFrame:
    """
    Convert match-centric data (1 row per match) into team-centric data 
    (2 rows per match — one per team's perspective).
    
    This makes rolling computations natural because each team's history 
    becomes a contiguous chronological series.
    """
    # Home team perspective
    home_view = pd.DataFrame({
        'match_id': matches['id'],
        'date': matches['date'],
        'team_api_id': matches['home_team_api_id'],
        'is_home': True,
        'goals_scored': matches['home_team_goal'],
        'goals_conceded': matches['away_team_goal'],
    })
    home_view['won'] = (home_view['goals_scored'] > home_view['goals_conceded']).astype(int)
    home_view['drew'] = (home_view['goals_scored'] == home_view['goals_conceded']).astype(int)
    home_view['lost'] = (home_view['goals_scored'] < home_view['goals_conceded']).astype(int)
    
    # Away team perspective — same but with home/away flipped
    away_view = pd.DataFrame({
        'match_id': matches['id'],
        'date': matches['date'],
        'team_api_id': matches['away_team_api_id'],
        'is_home': False,
        'goals_scored': matches['away_team_goal'],
        'goals_conceded': matches['home_team_goal'],
    })
    away_view['won'] = (away_view['goals_scored'] > away_view['goals_conceded']).astype(int)
    away_view['drew'] = (away_view['goals_scored'] == away_view['goals_conceded']).astype(int)
    away_view['lost'] = (away_view['goals_scored'] < away_view['goals_conceded']).astype(int)
    
    team_centric = pd.concat([home_view, away_view], ignore_index=True)
    team_centric = team_centric.sort_values(['team_api_id', 'date']).reset_index(drop=True)
    return team_centric


def compute_rolling_form(matches: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    """
    Add rolling-form features for the home and away team in each match.
    
    For each match, computes (using only matches BEFORE this one):
      - {home,away}_form_wins:    wins in last `window` matches
      - {home,away}_form_draws:   draws in last `window` matches
      - {home,away}_form_losses:  losses in last `window` matches
      - {home,away}_form_gs_avg:  average goals scored per match
      - {home,away}_form_gc_avg:  average goals conceded per match
    
    Args:
        matches: DataFrame with target already derived.
        window: How many previous matches to roll over (default 5).
    
    Returns:
        DataFrame with 10 new feature columns. Rows without enough history are dropped.
    """
    logger.info(f"Computing rolling form features (window={window})")
    
    # Step 1: Convert to team-centric view
    team_centric = _build_team_centric_view(matches)
    logger.info(f"  Team-centric view: {len(team_centric):,} rows")
    
    # Step 2: Compute rolling stats per team, SHIFTED by 1 (no leakage)
    # The .shift(1) is critical — it means "everything BEFORE this row", excluding self
    # min_periods=window means we require a FULL window of history (drops cold-start)
    grouped = team_centric.groupby('team_api_id', group_keys=False)
    
    team_centric['form_wins'] = grouped['won'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=window).sum()
    )
    team_centric['form_draws'] = grouped['drew'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=window).sum()
    )
    team_centric['form_losses'] = grouped['lost'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=window).sum()
    )
    team_centric['form_gs_avg'] = grouped['goals_scored'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=window).mean()
    )
    team_centric['form_gc_avg'] = grouped['goals_conceded'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=window).mean()
    )
    
    # Step 3: Merge back to match-centric
    # We have team_centric with 2 rows per match (home + away). 
    # Pivot back: for each match_id, get home stats and away stats separately.
    home_stats = team_centric[team_centric['is_home']].set_index('match_id')[
        ['form_wins', 'form_draws', 'form_losses', 'form_gs_avg', 'form_gc_avg']
    ].add_prefix('home_')
    
    away_stats = team_centric[~team_centric['is_home']].set_index('match_id')[
        ['form_wins', 'form_draws', 'form_losses', 'form_gs_avg', 'form_gc_avg']
    ].add_prefix('away_')
    
    matches = matches.copy()
    matches = matches.merge(home_stats, left_on='id', right_index=True, how='left')
    matches = matches.merge(away_stats, left_on='id', right_index=True, how='left')
    
    # Step 4: Drop matches without full form history (cold-start)
    initial_count = len(matches)
    form_cols = [c for c in matches.columns if c.endswith(('_wins', '_draws', '_losses', '_gs_avg', '_gc_avg'))]
    matches = matches.dropna(subset=form_cols)
    dropped = initial_count - len(matches)
    logger.info(f"  Dropped {dropped:,} matches without full form history (cold-start)")
    logger.info(f"  Remaining: {len(matches):,} matches with form features")
    
    return matches

In [9]:
matches_with_form = compute_rolling_form(matches, window=5)

print("\n--- Form feature columns added ---")
form_cols = [c for c in matches_with_form.columns if 'form' in c]
print(form_cols)

print("\n--- Sample with form features ---")
sample_cols = ['date', 'home_team_api_id', 'away_team_api_id', 
               'home_form_wins', 'home_form_gs_avg', 'home_form_gc_avg',
               'away_form_wins', 'away_form_gs_avg', 'away_form_gc_avg',
               'outcome']
print(matches_with_form[sample_cols].head(10))

print("\n--- Form feature statistics ---")
print(matches_with_form[form_cols].describe().round(2))

print(f"\n--- Class distribution after dropping cold-start matches ---")
print(matches_with_form['outcome'].value_counts(normalize=True).round(3))

2026-04-20 02:23:15 | INFO     | feature_engineering_dev | Computing rolling form features (window=5)
2026-04-20 02:23:15 | INFO     | feature_engineering_dev |   Team-centric view: 51,958 rows
2026-04-20 02:23:15 | INFO     | feature_engineering_dev |   Dropped 1,018 matches without full form history (cold-start)
2026-04-20 02:23:15 | INFO     | feature_engineering_dev |   Remaining: 24,961 matches with form features

--- Form feature columns added ---
['home_form_wins', 'home_form_draws', 'home_form_losses', 'home_form_gs_avg', 'home_form_gc_avg', 'away_form_wins', 'away_form_draws', 'away_form_losses', 'away_form_gs_avg', 'away_form_gc_avg']

--- Sample with form features ---
          date  home_team_api_id  away_team_api_id  home_form_wins  \
87  2008-08-16              9824             10179             1.0   
102 2008-08-17             10199             10192             0.0   
103 2008-08-17             10243              9931             3.0   
147 2008-08-23              9931

In [10]:
# Confirm wins + draws + losses = 5 for every row (no math errors)
home_total = matches_with_form[['home_form_wins', 'home_form_draws', 'home_form_losses']].sum(axis=1)
away_total = matches_with_form[['away_form_wins', 'away_form_draws', 'away_form_losses']].sum(axis=1)

print(f"Home rows where wins+draws+losses == 5: {(home_total == 5).sum()} / {len(matches_with_form)}")
print(f"Away rows where wins+draws+losses == 5: {(away_total == 5).sum()} / {len(matches_with_form)}")

Home rows where wins+draws+losses == 5: 24961 / 24961
Away rows where wins+draws+losses == 5: 24961 / 24961


### **Function 4 — compute_head_to_head()**

In [11]:
def compute_head_to_head(matches: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    """
    Add head-to-head features for each match.
    
    For each match, looks at the last `window` meetings between this exact 
    pair of teams (in any home/away configuration) and computes:
      - h2h_home_wins:   wins by the team currently at home
      - h2h_draws:       draws between this pair
      - h2h_away_wins:   wins by the team currently at away
      - h2h_n_meetings:  how many prior meetings exist (0 to window)
    
    Pairings with no prior meetings get all-zero h2h features. The 
    h2h_n_meetings feature lets the model learn to weight H2H by reliability.
    
    Args:
        matches: DataFrame with target already derived.
        window: Max number of prior meetings to consider (default 5).
    
    Returns:
        DataFrame with 4 new feature columns.
    """
    logger.info(f"Computing head-to-head features (window={window})")
    
    matches = matches.copy()
    
    # Step 1: Create normalized team_pair key
    # Sorted tuple ensures (A,B) and (B,A) become the same key
    matches['team_pair'] = matches.apply(
        lambda r: tuple(sorted([r['home_team_api_id'], r['away_team_api_id']])),
        axis=1
    )
    
    # Step 2: For each row, compute "did the currently-home-team win this match?"
    # We need this so when we roll over history, we count from a consistent perspective
    matches['curr_home_won'] = (matches['home_team_goal'] > matches['away_team_goal']).astype(int)
    matches['curr_drew'] = (matches['home_team_goal'] == matches['away_team_goal']).astype(int)
    matches['curr_away_won'] = (matches['home_team_goal'] < matches['away_team_goal']).astype(int)
    
    # Step 3: Sort by pair and date so we can roll within each pair's history
    matches = matches.sort_values(['team_pair', 'date']).reset_index(drop=True)
    
    # Step 4: For each pair, compute rolling sum of past meetings
    # BUT — and this is the tricky part — the "perspective" might flip from match to match
    # Example: in 2010, Team A was home; in 2012, Team B was home (still same pair)
    # So we can't just roll on curr_home_won — we need to flip when teams swap
    #
    # Trick: compute "did team_with_lower_id win?" — this is invariant to home/away swap
    matches['lower_id'] = matches['team_pair'].apply(lambda p: p[0])
    matches['lower_id_won'] = np.where(
        matches['home_team_api_id'] == matches['lower_id'],
        matches['curr_home_won'],
        matches['curr_away_won']
    )
    matches['higher_id_won'] = np.where(
        matches['home_team_api_id'] == matches['lower_id'],
        matches['curr_away_won'],
        matches['curr_home_won']
    )
    
    # Step 5: Roll within each pair, shifted by 1 (no leakage)
    grouped = matches.groupby('team_pair', group_keys=False)
    matches['h2h_lower_wins_raw'] = grouped['lower_id_won'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=1).sum()
    )
    matches['h2h_higher_wins_raw'] = grouped['higher_id_won'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=1).sum()
    )
    matches['h2h_draws'] = grouped['curr_drew'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=1).sum()
    )
    # Count how many meetings the rolling window actually had
    matches['h2h_n_meetings'] = grouped['curr_drew'].apply(
        lambda s: s.shift(1).rolling(window, min_periods=1).count()
    )
    
    # Step 6: Translate from "lower/higher id" back to "current home/away team" perspective
    matches['h2h_home_wins'] = np.where(
        matches['home_team_api_id'] == matches['lower_id'],
        matches['h2h_lower_wins_raw'],
        matches['h2h_higher_wins_raw']
    )
    matches['h2h_away_wins'] = np.where(
        matches['home_team_api_id'] == matches['lower_id'],
        matches['h2h_higher_wins_raw'],
        matches['h2h_lower_wins_raw']
    )
    
    # Step 7: Fill NaN (pairs with zero prior meetings) with 0
    h2h_cols = ['h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_n_meetings']
    matches[h2h_cols] = matches[h2h_cols].fillna(0)
    
    # Step 8: Drop intermediate scratch columns
    scratch_cols = ['team_pair', 'curr_home_won', 'curr_drew', 'curr_away_won',
                    'lower_id', 'lower_id_won', 'higher_id_won',
                    'h2h_lower_wins_raw', 'h2h_higher_wins_raw']
    matches = matches.drop(columns=scratch_cols)
    
    # Re-sort by date for downstream consistency
    matches = matches.sort_values('date').reset_index(drop=True)
    
    # Logging
    n_no_history = (matches['h2h_n_meetings'] == 0).sum()
    logger.info(f"  Matches with zero prior H2H meetings: {n_no_history:,} ({100*n_no_history/len(matches):.1f}%)")
    logger.info(f"  Mean prior H2H meetings: {matches['h2h_n_meetings'].mean():.2f}")
    logger.info(f"  Computed H2H features for {len(matches):,} matches")
    
    return matches

In [12]:
matches_with_h2h = compute_head_to_head(matches_with_form, window=5)

print("\n--- H2H feature columns ---")
h2h_cols = [c for c in matches_with_h2h.columns if 'h2h' in c]
print(h2h_cols)

print("\n--- Sample with H2H features (matches with 3+ prior meetings) ---")
sample_cols = ['date', 'home_team_api_id', 'away_team_api_id',
               'h2h_home_wins', 'h2h_draws', 'h2h_away_wins', 'h2h_n_meetings',
               'outcome']
print(matches_with_h2h[matches_with_h2h['h2h_n_meetings'] >= 3][sample_cols].head(10))

print("\n--- H2H feature statistics ---")
print(matches_with_h2h[h2h_cols].describe().round(2))

print(f"\n--- Distribution of n_meetings ---")
print(matches_with_h2h['h2h_n_meetings'].value_counts().sort_index())

2026-04-20 02:48:46 | INFO     | feature_engineering_dev | Computing head-to-head features (window=5)
2026-04-20 02:48:49 | INFO     | feature_engineering_dev |   Matches with zero prior H2H meetings: 3,510 (14.1%)
2026-04-20 02:48:49 | INFO     | feature_engineering_dev |   Mean prior H2H meetings: 3.22
2026-04-20 02:48:49 | INFO     | feature_engineering_dev |   Computed H2H features for 24,961 matches

--- H2H feature columns ---
['h2h_draws', 'h2h_n_meetings', 'h2h_home_wins', 'h2h_away_wins']

--- Sample with H2H features (matches with 3+ prior meetings) ---
           date  home_team_api_id  away_team_api_id  h2h_home_wins  h2h_draws  \
2283 2009-04-18             10243             10179            3.0        0.0   
2303 2009-04-19              7955              9931            1.0        0.0   
2311 2009-04-19             10192              9956            1.0        1.0   
2338 2009-04-22             10199             10179            0.0        3.0   
2463 2009-05-02          

In [15]:
total = matches_with_h2h[['h2h_home_wins', 'h2h_draws', 'h2h_away_wins']].sum(axis=1)
print(f"Rows where h2h sums equal n_meetings: {(total == matches_with_h2h['h2h_n_meetings']).sum()} / {len(matches_with_h2h)}")
   

Rows where h2h sums equal n_meetings: 24961 / 24961


### **Function 5 — attach_fifa_features()**

In [16]:
def attach_fifa_features(matches: pd.DataFrame, team_attrs: pd.DataFrame) -> pd.DataFrame:
    """
    Add FIFA team attribute features for both home and away teams.
    
    Uses a time-aware (backward) merge: for each match, finds the most recent 
    FIFA snapshot for each team that occurred BEFORE the match date. This 
    prevents target leakage from using future FIFA ratings.
    
    Adds 18 features: 9 FIFA attributes × 2 teams (home and away).
    Missing values (e.g., teams with no FIFA snapshot before the match date)
    are imputed with column means.
    
    Args:
        matches: DataFrame with date column.
        team_attrs: DataFrame with team_api_id, date, and 9 FIFA columns.
    
    Returns:
        DataFrame with 18 new feature columns, all non-null.
    """
    logger.info("Attaching FIFA team attribute features (time-aware join)")
    
    matches = matches.copy()
    
    # The 9 numeric FIFA columns we care about
    fifa_numeric_cols = [
        'buildUpPlaySpeed', 'buildUpPlayDribbling', 'buildUpPlayPassing',
        'chanceCreationPassing', 'chanceCreationCrossing', 'chanceCreationShooting',
        'defencePressure', 'defenceAggression', 'defenceTeamWidth',
    ]
    
    # Slim down team_attrs to only what we need + sort by date (required by merge_asof)
    attrs_slim = team_attrs[['team_api_id', 'date'] + fifa_numeric_cols].copy()
    attrs_slim = attrs_slim.sort_values('date').reset_index(drop=True)
    
    # merge_asof requires the LEFT table sorted by `on` column too
    matches = matches.sort_values('date').reset_index(drop=True)
    
    # --- Home team FIFA join ---
    # Rename team col on the right side to match home_team_api_id
    home_attrs = attrs_slim.rename(columns={'team_api_id': 'home_team_api_id'})
    matches = pd.merge_asof(
        left=matches,
        right=home_attrs,
        on='date',
        by='home_team_api_id',
        direction='backward',
    )
    # Rename the joined FIFA columns with home_ prefix
    matches = matches.rename(columns={c: f'home_{c}' for c in fifa_numeric_cols})
    
    # --- Away team FIFA join ---
    away_attrs = attrs_slim.rename(columns={'team_api_id': 'away_team_api_id'})
    matches = pd.merge_asof(
        left=matches,
        right=away_attrs,
        on='date',
        by='away_team_api_id',
        direction='backward',
    )
    matches = matches.rename(columns={c: f'away_{c}' for c in fifa_numeric_cols})
    
    # --- Impute missing values with column means ---
    fifa_feature_cols = (
        [f'home_{c}' for c in fifa_numeric_cols] + 
        [f'away_{c}' for c in fifa_numeric_cols]
    )
    
    n_missing_before = matches[fifa_feature_cols].isna().sum().sum()
    logger.info(f"  Missing FIFA values before imputation: {n_missing_before:,}")
    
    # Impute each column with its own mean (computed across all matches)
    for col in fifa_feature_cols:
        if matches[col].isna().any():
            mean_val = matches[col].mean()
            matches[col] = matches[col].fillna(mean_val)
    
    n_missing_after = matches[fifa_feature_cols].isna().sum().sum()
    logger.info(f"  Missing FIFA values after imputation: {n_missing_after:,}")
    logger.info(f"  Added {len(fifa_feature_cols)} FIFA features")
    
    return matches

In [17]:
matches_with_fifa = attach_fifa_features(matches_with_h2h, team_attrs)

print("\n--- FIFA feature columns added (showing 6 of 18) ---")
fifa_cols = [c for c in matches_with_fifa.columns if any(k in c for k in 
    ['buildUp', 'chanceCreation', 'defence']) and (c.startswith('home_') or c.startswith('away_'))]
print(f"Total FIFA features: {len(fifa_cols)}")
print(fifa_cols[:6], "...")

print("\n--- Sample with FIFA features ---")
sample_cols = ['date', 'home_team_api_id', 'home_buildUpPlaySpeed', 
               'home_chanceCreationShooting', 'home_defencePressure',
               'away_buildUpPlaySpeed', 'away_chanceCreationShooting', 
               'away_defencePressure', 'outcome']
print(matches_with_fifa[sample_cols].head(10))

print("\n--- Statistics on a few FIFA features ---")
print(matches_with_fifa[['home_buildUpPlaySpeed', 'home_chanceCreationShooting',
                          'home_defencePressure', 'away_buildUpPlaySpeed']].describe().round(2))

print(f"\n--- Total matches with all features: {len(matches_with_fifa):,} ---")
print(f"--- Total columns: {len(matches_with_fifa.columns)} ---")

# Critical check: NO nulls anywhere in our feature space
all_feature_cols = [c for c in matches_with_fifa.columns if c.startswith(('home_', 'away_', 'h2h_')) 
                    and 'team_api_id' not in c and 'team_goal' not in c]
n_nulls = matches_with_fifa[all_feature_cols].isna().sum().sum()
print(f"--- Total nulls in feature columns: {n_nulls} (should be 0) ---")

2026-04-20 02:58:07 | INFO     | feature_engineering_dev | Attaching FIFA team attribute features (time-aware join)
2026-04-20 02:58:07 | INFO     | feature_engineering_dev |   Missing FIFA values before imputation: 124,299
2026-04-20 02:58:07 | INFO     | feature_engineering_dev |   Missing FIFA values after imputation: 0
2026-04-20 02:58:07 | INFO     | feature_engineering_dev |   Added 18 FIFA features

--- FIFA feature columns added (showing 6 of 18) ---
Total FIFA features: 18
['home_buildUpPlaySpeed', 'home_buildUpPlayDribbling', 'home_buildUpPlayPassing', 'home_chanceCreationPassing', 'home_chanceCreationCrossing', 'home_chanceCreationShooting'] ...

--- Sample with FIFA features ---
        date  home_team_api_id  home_buildUpPlaySpeed  \
0 2008-08-16              9824              52.423458   
1 2008-08-17             10243              52.423458   
2 2008-08-17             10199              52.423458   
3 2008-08-23              9956              52.423458   
4 2008-08-23   

### **Function 6 — chronological_split()**

In [18]:
def chronological_split(
    matches: pd.DataFrame,
    train_seasons: list[str],
    test_seasons: list[str],
    production_seasons: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split matches into train/test/production sets based on season.
    
    Chronological splitting (not random) is required because matches are 
    time-ordered events. Random splits would leak future information into 
    training, producing inflated test scores that don't reflect deployment 
    performance.
    
    Args:
        matches: Full matches DataFrame with 'season' column.
        train_seasons: List of season strings (e.g., ['2008/2009', '2009/2010']).
        test_seasons: List of season strings for held-out evaluation.
        production_seasons: List of season strings for drift simulation.
    
    Returns:
        Tuple of (train_df, test_df, production_df).
    """
    logger.info("Splitting matches chronologically by season")
    logger.info(f"  Train seasons:      {train_seasons}")
    logger.info(f"  Test seasons:       {test_seasons}")
    logger.info(f"  Production seasons: {production_seasons}")
    
    # Validate that all specified seasons exist in the data
    available_seasons = set(matches['season'].unique())
    requested_seasons = set(train_seasons + test_seasons + production_seasons)
    missing = requested_seasons - available_seasons
    if missing:
        logger.error(f"Requested seasons not in data: {missing}")
        raise ValueError(f"Seasons not found: {missing}")
    
    # Validate no season appears in multiple splits (would cause leakage)
    all_split_seasons = train_seasons + test_seasons + production_seasons
    if len(all_split_seasons) != len(set(all_split_seasons)):
        raise ValueError("A season appears in multiple splits — would cause leakage")
    
    train_df = matches[matches['season'].isin(train_seasons)].copy()
    test_df = matches[matches['season'].isin(test_seasons)].copy()
    production_df = matches[matches['season'].isin(production_seasons)].copy()
    
    # Log split sizes and class distributions
    for name, df in [('Train', train_df), ('Test', test_df), ('Production', production_df)]:
        dist = df['outcome'].value_counts(normalize=True).round(3).to_dict()
        logger.info(f"  {name:12s}: {len(df):,} matches  |  outcome: {dist}")
    
    # Sanity check: total matches across splits should equal the input (or be less,
    # if some seasons in the data weren't requested)
    total_split = len(train_df) + len(test_df) + len(production_df)
    logger.info(f"  Total in splits: {total_split:,} / {len(matches):,} input matches")
    if total_split < len(matches):
        unused = len(matches) - total_split
        logger.warning(f"  {unused:,} matches not assigned to any split")
    
    return train_df, test_df, production_df

In [20]:
# Read split config from params.yaml (zero hardcoding — A1's commandment)
import yaml
with open(PROJECT_ROOT / "params.yaml") as f:
    params = yaml.safe_load(f)

train_df, test_df, production_df = chronological_split(
    matches_with_fifa,
    train_seasons=params['split']['train_seasons'],
    test_seasons=params['split']['test_seasons'],
    production_seasons=params['split']['production_seasons'],
)

print(f"\n--- Split sizes ---")
print(f"Train:      {len(train_df):,}  ({100*len(train_df)/len(matches_with_fifa):.1f}%)")
print(f"Test:       {len(test_df):,}  ({100*len(test_df)/len(matches_with_fifa):.1f}%)")
print(f"Production: {len(production_df):,}  ({100*len(production_df)/len(matches_with_fifa):.1f}%)")

print(f"\n--- Date ranges per split ---")
print(f"Train:      {train_df['date'].min().date()}  →  {train_df['date'].max().date()}")
print(f"Test:       {test_df['date'].min().date()}  →  {test_df['date'].max().date()}")
print(f"Production: {production_df['date'].min().date()}  →  {production_df['date'].max().date()}")

print(f"\n--- No date overlap check (critical) ---")
print(f"Train max < Test min:       {train_df['date'].max() < test_df['date'].min()}")
print(f"Test max < Production min:  {test_df['date'].max() < production_df['date'].min()}")

2026-04-20 03:03:05 | INFO     | feature_engineering_dev | Splitting matches chronologically by season
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Train seasons:      ['2008/2009', '2009/2010', '2010/2011', '2011/2012', '2012/2013', '2013/2014']
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Test seasons:       ['2014/2015']
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Production seasons: ['2015/2016']
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Train       : 18,429 matches  |  outcome: {'H': 0.464, 'A': 0.284, 'D': 0.252}
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Test        : 3,270 matches  |  outcome: {'H': 0.45, 'A': 0.296, 'D': 0.254}
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Production  : 3,262 matches  |  outcome: {'H': 0.44, 'A': 0.304, 'D': 0.256}
2026-04-20 03:03:05 | INFO     | feature_engineering_dev |   Total in splits: 24,961 / 24,961 input matches

--- Split sizes ---
Tr

### **main() and Save CSVs**


In [21]:
def main():
    """
    Orchestrate the full feature engineering pipeline:
    load → derive target → form → h2h → fifa → split → save.
    
    Reads paths and parameters from config.yaml and params.yaml so nothing 
    is hardcoded. Outputs CSVs to data/processed/ and data/production/.
    """
    logger.info("=" * 60)
    logger.info("FEATURE ENGINEERING PIPELINE — START")
    logger.info("=" * 60)
    
    # --- Load config + params ---
    with open(PROJECT_ROOT / "config.yaml") as f:
        config = yaml.safe_load(f)
    with open(PROJECT_ROOT / "params.yaml") as f:
        params = yaml.safe_load(f)
    
    db_path = PROJECT_ROOT / config['data']['raw_db']
    processed_dir = PROJECT_ROOT / config['data']['processed_dir']
    production_dir = PROJECT_ROOT / config['data']['production_dir']
    processed_dir.mkdir(parents=True, exist_ok=True)
    production_dir.mkdir(parents=True, exist_ok=True)
    
    # --- Run pipeline ---
    matches, teams, team_attrs = load_raw_data(db_path)
    matches = derive_target(matches)
    matches = compute_rolling_form(matches, window=params['features']['rolling_window_size'])
    matches = compute_head_to_head(matches, window=params['features']['h2h_window_size'])
    matches = attach_fifa_features(matches, team_attrs)
    
    train_df, test_df, production_df = chronological_split(
        matches,
        train_seasons=params['split']['train_seasons'],
        test_seasons=params['split']['test_seasons'],
        production_seasons=params['split']['production_seasons'],
    )
    
    # --- Save outputs ---
    train_path = processed_dir / "train.csv"
    test_path = processed_dir / "test.csv"
    production_path = production_dir / "season_2015_16.csv"
    
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    production_df.to_csv(production_path, index=False)
    
    logger.info(f"Saved train CSV:      {train_path} ({len(train_df):,} rows)")
    logger.info(f"Saved test CSV:       {test_path} ({len(test_df):,} rows)")
    logger.info(f"Saved production CSV: {production_path} ({len(production_df):,} rows)")
    
    # --- Save team lookup for the frontend/API ---
    team_lookup_path = processed_dir / "team_lookup.csv"
    teams_clean = teams[['team_api_id', 'team_long_name', 'team_short_name']].copy()
    teams_clean = teams_clean.dropna(subset=['team_long_name']).sort_values('team_long_name')
    teams_clean.to_csv(team_lookup_path, index=False)
    logger.info(f"Saved team lookup:    {team_lookup_path} ({len(teams_clean):,} teams)")
    
    logger.info("=" * 60)
    logger.info("FEATURE ENGINEERING PIPELINE — COMPLETE")
    logger.info("=" * 60)
    
    return train_df, test_df, production_df


# Run the pipeline end-to-end
train_df, test_df, production_df = main()

2026-04-20 03:06:04 | INFO     | feature_engineering_dev | ============================================================
2026-04-20 03:06:04 | INFO     | feature_engineering_dev | FEATURE ENGINEERING PIPELINE — START
2026-04-20 03:06:04 | INFO     | feature_engineering_dev | ============================================================
2026-04-20 03:06:04 | INFO     | feature_engineering_dev | Loading raw data from /home/muhammed786fiyas/Desktop/Projects/ml_ops/final_project/mlops_end_to_end_project/data/raw/database.sqlite
2026-04-20 03:06:04 | INFO     | feature_engineering_dev |   Loaded 25,979 matches
2026-04-20 03:06:04 | INFO     | feature_engineering_dev |   Loaded 299 teams
2026-04-20 03:06:04 | INFO     | feature_engineering_dev |   Loaded 1,458 team attribute snapshots
2026-04-20 03:06:04 | INFO     | feature_engineering_dev | Deriving target variable (H/D/A)
2026-04-20 03:06:04 | INFO     | feature_engineering_dev |   Outcome distribution: {'H': 0.459, 'A': 0.287, 'D': 0.254}


In [22]:
# Verify CSVs exist and look right
print("--- Files written ---")
for path_str in ["data/processed/train.csv", "data/processed/test.csv",
                 "data/production/season_2015_16.csv", "data/processed/team_lookup.csv"]:
    path = PROJECT_ROOT / path_str
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  ✓ {path_str:50s} ({size_mb:.2f} MB)")
    else:
        print(f"  ✗ MISSING: {path_str}")

print("\n--- train.csv preview ---")
train_check = pd.read_csv(PROJECT_ROOT / "data/processed/train.csv", nrows=3)
print(f"Columns ({len(train_check.columns)}): {train_check.columns.tolist()}")

print("\n--- team_lookup.csv preview ---")
lookup = pd.read_csv(PROJECT_ROOT / "data/processed/team_lookup.csv")
print(lookup.head(5))
print(f"Total teams in lookup: {len(lookup)}")

--- Files written ---
  ✓ data/processed/train.csv                           (5.11 MB)
  ✓ data/processed/test.csv                            (0.67 MB)
  ✓ data/production/season_2015_16.csv                 (0.65 MB)
  ✓ data/processed/team_lookup.csv                     (0.01 MB)

--- train.csv preview ---
Columns (44): ['id', 'country_id', 'league_id', 'season', 'stage', 'date', 'home_team_api_id', 'away_team_api_id', 'home_team_goal', 'away_team_goal', 'outcome', 'outcome_encoded', 'home_form_wins', 'home_form_draws', 'home_form_losses', 'home_form_gs_avg', 'home_form_gc_avg', 'away_form_wins', 'away_form_draws', 'away_form_losses', 'away_form_gs_avg', 'away_form_gc_avg', 'h2h_draws', 'h2h_n_meetings', 'h2h_home_wins', 'h2h_away_wins', 'home_buildUpPlaySpeed', 'home_buildUpPlayDribbling', 'home_buildUpPlayPassing', 'home_chanceCreationPassing', 'home_chanceCreationCrossing', 'home_chanceCreationShooting', 'home_defencePressure', 'home_defenceAggression', 'home_defenceTeamWidth', 'aw